# PyTorch CNN-LSTM Inference

This notebook loads a `.pt` checkpoint, chunks each video using the same strided/overlapping strategy as preprocessing, predicts each chunk, and applies the two-step video-level violence gate from `cnn_lstm_pytorch.py`.


In [ ]:
# Run this once if your notebook environment is missing PyTorch or inference dependencies:
# %pip install torch torchvision opencv-python matplotlib h5py

In [ ]:
from pathlib import Path
import sys
import importlib

start = Path.cwd().resolve()
BASE_DIR = next(
    (directory for directory in (start, *start.parents)
     if (directory / "src" / "cnn_lstm_pytorch.py").is_file()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("Start the notebook kernel inside the violence-movies project.")
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import cnn_lstm_pytorch as cnn_lstm
cnn_lstm = importlib.reload(cnn_lstm)

CLASS_NAMES = cnn_lstm.CLASS_NAMES
get_device = cnn_lstm.get_device
run_real_world_video_inference = cnn_lstm.run_real_world_video_inference

print(f"Base directory: {BASE_DIR}")
print(f"Device: {get_device()}")
print(f"cnn_lstm_pytorch module: {cnn_lstm.__file__}")
print(
    "Violence gate defaults: "
    f"presence={cnn_lstm.DEFAULT_VIOLENCE_PRESENCE_THRESHOLD}, "
    f"chunk={cnn_lstm.DEFAULT_VIOLENT_CHUNK_THRESHOLD}, "
    f"min_fraction={cnn_lstm.DEFAULT_MIN_VIOLENT_CHUNK_FRACTION}, "
    f"mean_margin={cnn_lstm.DEFAULT_VIOLENCE_MEAN_MARGIN}"
)
print(
    "High-severity gate defaults: "
    f"evidence={cnn_lstm.DEFAULT_HIGH_VIOLENCE_EVIDENCE_THRESHOLD}, "
    f"mean={cnn_lstm.DEFAULT_HIGH_VIOLENCE_MEAN_THRESHOLD}, "
    f"margin={cnn_lstm.DEFAULT_HIGH_VIOLENCE_MARGIN}"
)


## 1. Configure Inference

`MODEL_NAME` must match the architecture used when the checkpoint was saved in `train_cnn_lstm_pytorch.ipynb`.

In [ ]:
# Options: "3dcnnlstm", "c3dcnn", "lstm"
MODEL_NAME = "3dcnnlstm"

CHECKPOINTS = {
    "3dcnnlstm": BASE_DIR / "weights" / "3DCNNLSTM.pt",
    "c3dcnn": BASE_DIR / "weights" / "C3DCNN.pt",
    "lstm": BASE_DIR / "weights" / "LSTM.pt",
}

VIDEO_PATH = BASE_DIR / "data" / "streaming videos" / "filter_results"
# VIDEO_PATH = BASE_DIR / "data" / "streaming videos" / "filter_whole_movie"
CHECKPOINT_PATH = CHECKPOINTS[MODEL_NAME]

print(f"Video path: {VIDEO_PATH}")
print(f"Model: {MODEL_NAME}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## 2. Run Video-Level Prediction

In [ ]:
results = run_real_world_video_inference(
    video_path=VIDEO_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    class_names=CLASS_NAMES,
    freeze_backbone=True,
    batch_size=32,
    verbose=True,
)

results

## 3. Summary (only for whole movie analysis)

In [ ]:
import cv2
import numpy as np
import pandas as pd

# -----------------------------
# 1. Full streaming-video length information
# -----------------------------

video_lengths = {
    "John Wick": "01:41:12",
    "Hidden Man": "02:14:22",
    "Mission: Impossible -- Fallout": "02:27:41",
    "Ip Man": "01:46:21",
    "The Punisher": "00:54:02",
    "The Pursuit of Happyness": "01:57:27",
    "The Boys": "00:00:00",        # update if this title is included in your filtered clips
    "Game of Thrones": "00:00:00", # update if this title is included in your filtered clips
}

def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

video_length_seconds = {name: time_to_seconds(length) for name, length in video_lengths.items()}

# -----------------------------
# 2. Build clip-level table from PyTorch inference results
# -----------------------------

def get_video_length_sec(video_path):
    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    cap.release()
    return frame_count / fps if fps > 0 else 0.0

df = pd.DataFrame([
    {
        "Video": item["video_name"],
        "Video_Path": item["video_path"],
        "Prediction": item["video_pred_label"],
        "Prediction_ID": item["video_pred_id"],
        "Non-violence Prob": item["video_prob"][0],
        "Low-level Violence Prob": item["video_prob"][1],
        "High-level Violence Prob": item["video_prob"][2],
        "Video_Length_sec": get_video_length_sec(item["video_path"]),
    }
    for item in results
])

# -----------------------------
# 3. Identify streaming video name
# -----------------------------

def identify_streaming_video(video_name):
    name = video_name.lower().replace("_", " ").replace("-", " ")

    if "john" in name and "wick" in name:
        return "John Wick"
    if "hidden" in name and "man" in name:
        return "Hidden Man"
    if "mission" in name and ("impossible" in name or "fallout" in name):
        return "Mission: Impossible -- Fallout"
    if "ip" in name and "man" in name:
        return "Ip Man"
    if "punisher" in name:
        return "The Punisher"
    if "pursuit" in name and "happyness" in name:
        return "The Pursuit of Happyness"
    if "boys" in name:
        return "The Boys"
    if "game" in name and "thrones" in name:
        return "Game of Thrones"
    return "Unknown"

df["Streaming video"] = df["Video"].apply(identify_streaming_video)

unknown_df = df[df["Streaming video"] == "Unknown"]
print(f"Unknown rows: {len(unknown_df)}")
if len(unknown_df) > 0:
    display(unknown_df[["Video", "Prediction"]].head(20))

# -----------------------------
# 4. Keep only low/high violence and summarise by streaming video
# -----------------------------

violent_df = df[df["Prediction"].isin(["low-level violence", "high-level violence"])].copy()

summary_rows = []
for movie, full_length in video_lengths.items():
    movie_df = violent_df[violent_df["Streaming video"] == movie]
    low_df = movie_df[movie_df["Prediction"] == "low-level violence"]
    high_df = movie_df[movie_df["Prediction"] == "high-level violence"]

    low_clips = len(low_df)
    high_clips = len(high_df)
    total_violent_clips = low_clips + high_clips
    low_duration = low_df["Video_Length_sec"].sum()
    high_duration = high_df["Video_Length_sec"].sum()
    total_duration = low_duration + high_duration
    full_video_length_sec = video_length_seconds[movie]

    summary_rows.append({
        "Streaming video": movie,
        "Video length": full_length,
        "Video length (s)": full_video_length_sec,
        "Total violent clips": total_violent_clips,
        "Low-level violence clips": low_clips,
        "High-level violence clips": high_clips,
        "Total violence duration (s)": int(round(total_duration)),
        "Low-level Violence duration (s)": int(round(low_duration)),
        "High-level Violence duration (s)": int(round(high_duration)),
        "Violence Ratio": total_duration / full_video_length_sec if full_video_length_sec else np.nan,
        "Low-level Violence Ratio": low_duration / full_video_length_sec if full_video_length_sec else np.nan,
        "High-level Violence Ratio": high_duration / full_video_length_sec if full_video_length_sec else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df